# ENARES 2024 CRS04 — Stage 03
## Notebook 06 — Acumulación y coocurrencia de violencias

Traducción del bloque de creación de variables del archivo SPSS:

`11_CRS04_3.5 Acumulación de violencias_ver4 (1)(2).sps`

Crea ocho indicadores dicotómicos de coocurrencia entre hogar y escuela. Los tabulados ponderados con diseño muestral complejo se realizan posteriormente en R.

In [1]:
!pip install -q google-cloud-bigquery pandas pandas-gbq pyarrow db-dtypes

In [2]:
from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

auth.authenticate_user()
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
PROJECT_ID = "enares-2024-crs04"
LOCATION = "US"
EXPECTED_ROWS = 18807

ROOT_DRIVE = Path("/content/drive/MyDrive/ENARES_2024_PROJECT")
LOG_DIR = ROOT_DRIVE / "05Resultados" / "logs" / "stage03"
SQL_DIR = ROOT_DRIVE / "02SQL"

for directory in [LOG_DIR, SQL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_UTC = datetime.now(timezone.utc).isoformat()
print("PROJECT_ID:", PROJECT_ID)
print("RUN_UTC:", RUN_UTC)

PROJECT_ID: enares-2024-crs04
RUN_UTC: 2026-07-17T04:41:05.218656+00:00


In [4]:
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

print("Target table:", A)

Target table: enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents


In [5]:
required_inputs = [
    "INDICADOR_8_3_6",
    "INDICADOR_8_3_9",
    "VP_o_VF_HOGAR",
    "VP_o_VF_ESCUELA",
    "VP_HOGAR",
    "VF_HOGAR",
    "VP_ESCUELA",
    "VF_ESCUELA",
    "VS_12M",
]

table_before = client.get_table(A)
existing_columns = {field.name for field in table_before.schema}
missing_inputs = sorted(set(required_inputs) - existing_columns)

if missing_inputs:
    raise RuntimeError(
        "No se puede ejecutar la sección 3.5. Faltan variables: "
        + ", ".join(missing_inputs)
    )

if table_before.num_rows != EXPECTED_ROWS:
    raise RuntimeError(
        f"La tabla contiene {table_before.num_rows:,} filas; "
        f"se esperaban {EXPECTED_ROWS:,}."
    )

print("Prerequisitos aprobados.")
print("Filas:", table_before.num_rows)
print("Variables fuente verificadas:", len(required_inputs))

Prerequisitos aprobados.
Filas: 18807
Variables fuente verificadas: 9


In [6]:
output_columns = [
    "PV_hogar_escuela1",
    "PV_hogar_escuela",
    "PV_VP_hogar_escuela",
    "PV_VF_hogar_escuela",
    "PV_VP_hogar_VF_escuela",
    "PV_VF_hogar_VP_escuela",
    "PV_VP_VF_hogar_escuela",
    "PV_VP_VF_hogar_escuela_VS",
]

columns_to_replace = [c for c in output_columns if c in existing_columns]
source_select = (
    "* EXCEPT(" + ", ".join(f"`{c}`" for c in columns_to_replace) + ")"
    if columns_to_replace else "*"
)

sql_35 = f"""
CREATE OR REPLACE TABLE `{A}` AS
SELECT
  {source_select},
  CASE WHEN INDICADOR_8_3_6 = 1 AND INDICADOR_8_3_9 = 1 THEN 1 ELSE 0 END AS PV_hogar_escuela1,
  CASE WHEN VP_o_VF_HOGAR = 1 AND VP_o_VF_ESCUELA = 1 THEN 1 ELSE 0 END AS PV_hogar_escuela,
  CASE WHEN VP_HOGAR = 1 AND VP_ESCUELA = 1 THEN 1 ELSE 0 END AS PV_VP_hogar_escuela,
  CASE WHEN VF_HOGAR = 1 AND VF_ESCUELA = 1 THEN 1 ELSE 0 END AS PV_VF_hogar_escuela,
  CASE WHEN VP_HOGAR = 1 AND VF_ESCUELA = 1 THEN 1 ELSE 0 END AS PV_VP_hogar_VF_escuela,
  CASE WHEN VF_HOGAR = 1 AND VP_ESCUELA = 1 THEN 1 ELSE 0 END AS PV_VF_hogar_VP_escuela,
  CASE WHEN VP_HOGAR = 1 AND VF_HOGAR = 1 AND VP_ESCUELA = 1 AND VF_ESCUELA = 1 THEN 1 ELSE 0 END AS PV_VP_VF_hogar_escuela,
  CASE WHEN VP_HOGAR = 1 AND VF_HOGAR = 1 AND VP_ESCUELA = 1 AND VF_ESCUELA = 1 AND VS_12M = 1 THEN 1 ELSE 0 END AS PV_VP_VF_hogar_escuela_VS
FROM `{A}`
"""

sql_path = SQL_DIR / "stage3_35_acumulacion_violencias.sql"
sql_path.write_text(sql_35, encoding="utf-8")
client.query(sql_35, location=LOCATION).result()

print("Indicadores 3.5 creados correctamente.")
print("SQL guardado en:", sql_path)

Indicadores 3.5 creados correctamente.
SQL guardado en: /content/drive/MyDrive/ENARES_2024_PROJECT/02SQL/stage3_35_acumulacion_violencias.sql


In [7]:
table_after = client.get_table(A)
columns_after = {field.name for field in table_after.schema}
missing_outputs = sorted(set(output_columns) - columns_after)

if missing_outputs:
    raise RuntimeError("Faltan columnas de salida: " + ", ".join(missing_outputs))
if table_after.num_rows != EXPECTED_ROWS:
    raise RuntimeError(
        f"La tabla terminó con {table_after.num_rows:,} filas; se esperaban {EXPECTED_ROWS:,}."
    )

print("Esquema y conteo de filas aprobados.")
print("Indicadores creados:", len(output_columns))

Esquema y conteo de filas aprobados.
Indicadores creados: 8


In [8]:
validation_parts = []
for variable in output_columns:
    query = f"""
    SELECT
      '{variable}' AS variable,
      COUNT(*) AS total_rows,
      COUNTIF(`{variable}` IS NULL) AS null_values,
      COUNTIF(`{variable}` NOT IN (0, 1)) AS invalid_values,
      COUNTIF(`{variable}` = 0) AS zero_values,
      COUNTIF(`{variable}` = 1) AS one_values
    FROM `{A}`
    """
    validation_parts.append(client.query(query, location=LOCATION).result().to_dataframe())

validation_35 = pd.concat(validation_parts, ignore_index=True)
validation_path = LOG_DIR / "stage3_35_domain_validation.csv"
validation_35.to_csv(validation_path, index=False)
display(validation_35)

if (validation_35["total_rows"] != EXPECTED_ROWS).any():
    raise RuntimeError("Algún indicador alteró el universo analítico.")
if (validation_35["null_values"] > 0).any():
    raise RuntimeError("Algún indicador contiene valores NULL.")
if (validation_35["invalid_values"] > 0).any():
    raise RuntimeError("Algún indicador contiene valores fuera de 0/1.")

print("Validación de dominio aprobada.")

,variable,total_rows,null_values,invalid_values,zero_values,one_values
0,PV_hogar_escuela1,18807,0,0,13748,5059
1,PV_hogar_escuela,18807,0,0,14869,3938
2,PV_VP_hogar_escuela,18807,0,0,15560,3247
3,PV_VF_hogar_escuela,18807,0,0,17972,835
4,PV_VP_hogar_VF_escuela,18807,0,0,17607,1200
5,PV_VF_hogar_VP_escuela,18807,0,0,16793,2014
6,PV_VP_VF_hogar_escuela,18807,0,0,18236,571
7,PV_VP_VF_hogar_escuela_VS,18807,0,0,18492,315


Validación de dominio aprobada.


In [9]:
consistency_35 = client.query(f"""
SELECT
  COUNT(*) AS total_rows,
  COUNTIF(PV_hogar_escuela1 != CASE WHEN INDICADOR_8_3_6 = 1 AND INDICADOR_8_3_9 = 1 THEN 1 ELSE 0 END) AS bad_PV_hogar_escuela1,
  COUNTIF(PV_hogar_escuela != CASE WHEN VP_o_VF_HOGAR = 1 AND VP_o_VF_ESCUELA = 1 THEN 1 ELSE 0 END) AS bad_PV_hogar_escuela,
  COUNTIF(PV_VP_hogar_escuela != CASE WHEN VP_HOGAR = 1 AND VP_ESCUELA = 1 THEN 1 ELSE 0 END) AS bad_PV_VP_hogar_escuela,
  COUNTIF(PV_VF_hogar_escuela != CASE WHEN VF_HOGAR = 1 AND VF_ESCUELA = 1 THEN 1 ELSE 0 END) AS bad_PV_VF_hogar_escuela,
  COUNTIF(PV_VP_hogar_VF_escuela != CASE WHEN VP_HOGAR = 1 AND VF_ESCUELA = 1 THEN 1 ELSE 0 END) AS bad_PV_VP_hogar_VF_escuela,
  COUNTIF(PV_VF_hogar_VP_escuela != CASE WHEN VF_HOGAR = 1 AND VP_ESCUELA = 1 THEN 1 ELSE 0 END) AS bad_PV_VF_hogar_VP_escuela,
  COUNTIF(PV_VP_VF_hogar_escuela != CASE WHEN VP_HOGAR = 1 AND VF_HOGAR = 1 AND VP_ESCUELA = 1 AND VF_ESCUELA = 1 THEN 1 ELSE 0 END) AS bad_PV_VP_VF_hogar_escuela,
  COUNTIF(PV_VP_VF_hogar_escuela_VS != CASE WHEN VP_HOGAR = 1 AND VF_HOGAR = 1 AND VP_ESCUELA = 1 AND VF_ESCUELA = 1 AND VS_12M = 1 THEN 1 ELSE 0 END) AS bad_PV_VP_VF_hogar_escuela_VS
FROM `{A}`
""", location=LOCATION).result().to_dataframe()

consistency_path = LOG_DIR / "stage3_35_consistency_validation.csv"
consistency_35.to_csv(consistency_path, index=False)
display(consistency_35)

bad_columns = [
    c for c in consistency_35.columns
    if c.startswith("bad_") and int(consistency_35.iloc[0][c]) > 0
]
if bad_columns:
    raise RuntimeError("Inconsistencias lógicas en: " + ", ".join(bad_columns))

print("Consistencia lógica aprobada para los ocho indicadores.")

,total_rows,bad_PV_hogar_escuela1,bad_PV_hogar_escuela,bad_PV_VP_hogar_escuela,bad_PV_VF_hogar_escuela,bad_PV_VP_hogar_VF_escuela,bad_PV_VF_hogar_VP_escuela,bad_PV_VP_VF_hogar_escuela,bad_PV_VP_VF_hogar_escuela_VS
0,18807,0,0,0,0,0,0,0,0


Consistencia lógica aprobada para los ocho indicadores.


In [10]:
distribution_parts = []
for variable in output_columns:
    query = f"""
    SELECT
      '{variable}' AS variable,
      `{variable}` AS value,
      COUNT(*) AS n,
      ROUND(100 * SAFE_DIVIDE(COUNT(*), SUM(COUNT(*)) OVER ()), 4) AS percent_unweighted
    FROM `{A}`
    GROUP BY `{variable}`
    """
    distribution_parts.append(client.query(query, location=LOCATION).result().to_dataframe())

distribution_35 = pd.concat(distribution_parts, ignore_index=True)
distribution_35 = distribution_35.sort_values(["variable", "value"]).reset_index(drop=True)
distribution_path = LOG_DIR / "stage3_35_distribution_unweighted_qa.csv"
distribution_35.to_csv(distribution_path, index=False)
display(distribution_35)
print("Distribuciones QA guardadas en:", distribution_path)

,variable,value,n,percent_unweighted
0,PV_VF_hogar_VP_escuela,0,16793,89.2912
1,PV_VF_hogar_VP_escuela,1,2014,10.7088
2,PV_VF_hogar_escuela,0,17972,95.5602
3,PV_VF_hogar_escuela,1,835,4.4398
4,PV_VP_VF_hogar_escuela,0,18236,96.9639
5,PV_VP_VF_hogar_escuela,1,571,3.0361
6,PV_VP_VF_hogar_escuela_VS,0,18492,98.3251
7,PV_VP_VF_hogar_escuela_VS,1,315,1.6749
8,PV_VP_hogar_VF_escuela,0,17607,93.6194
9,PV_VP_hogar_VF_escuela,1,1200,6.3806


Distribuciones QA guardadas en: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/stage03/stage3_35_distribution_unweighted_qa.csv


In [11]:
closure_35 = pd.DataFrame([{
    "run_utc": RUN_UTC,
    "table": A,
    "expected_rows": EXPECTED_ROWS,
    "actual_rows": table_after.num_rows,
    "indicators_created": len(output_columns),
    "domain_validation_pass": True,
    "consistency_validation_pass": True,
    "stage35_pass": True,
}])

closure_path = LOG_DIR / "stage3_35_closure.csv"
closure_35.to_csv(closure_path, index=False)
display(closure_35)
print("Stage 03 section 3.5 completed successfully.")
print("Closure log:", closure_path)

,run_utc,table,expected_rows,actual_rows,indicators_created,domain_validation_pass,consistency_validation_pass,stage35_pass
0,2026-07-17T04:41:05.218656+00:00,enares-2024-crs04.enares2024_crs04_analytical....,18807,18807,8,True,True,True


Stage 03 section 3.5 completed successfully.
Closure log: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/stage03/stage3_35_closure.csv
